# Match Cena ↔ Versículo

Sugere, pra cada versículo do capítulo, qual vídeo/imagem da sua biblioteca
(`pixabay_stock` ou `image-stock`) melhor combina — usando a sobreposição
entre as tags da lista fechada de Bíblia (`Tags_Biblia_PT`) já preenchidas
na planilha, e tags sugeridas por IA pra cada versículo especificamente.

**Não escreve nada automaticamente** — só gera uma lista pra você revisar.
Versículos sem nenhum vídeo com tag em comum aparecem marcados como
"sem opção", já com palavras-chave sugeridas prontas pra você usar na busca
ao vivo do Pixabay (mesmo diálogo que você já usa).

Funciona igual pra vídeo ou imagem — só muda `TIPO_FONTE` na Configuração.


In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  1. SETUP                                                        ║
# ╚══════════════════════════════════════════════════════════════════╝
!pip install -q -U groq gspread "mistralai>=1.2.0"

import shutil, sys, json
from pathlib import Path

from google.colab import drive, auth, userdata
from google.auth import default
import gspread
from groq import Groq
from mistralai.client import Mistral

try:
    drive.flush_and_unmount()
except Exception:
    pass
drive.mount('/content/drive', force_remount=True)

PASTA_DRIVE_RAIZ_MODULOS = "narrated_video"
PASTA_MODULOS = Path(f"/content/drive/MyDrive/{PASTA_DRIVE_RAIZ_MODULOS}/pipeline/modulos")
DESTINO = Path("/content/pipeline")
if PASTA_MODULOS.exists():
    if DESTINO.exists():
        shutil.rmtree(DESTINO)
    shutil.copytree(PASTA_MODULOS, DESTINO)
    print(f"✅ {len(list(DESTINO.glob('*.py')))} módulos copiados")
else:
    print(f"❌ Pasta de módulos não encontrada: {PASTA_MODULOS}")
if str(DESTINO) not in sys.path:
    sys.path.insert(0, str(DESTINO))

auth.authenticate_user()
creds, _ = default()
gc = gspread.authorize(creds)

GROQ_API_KEY = userdata.get("GROQ_KEY")
MISTRAL_API_KEY = userdata.get("MISTRAL_KEY")
groq_client = Groq(api_key=GROQ_API_KEY) if GROQ_API_KEY else None
mistral_client = Mistral(api_key=MISTRAL_API_KEY) if MISTRAL_API_KEY else None

print("✅ Setup concluído")
print(f"   Groq:    {'disponível' if groq_client else '❌ GROQ_KEY não encontrada'}")
print(f"   Mistral: {'disponível' if mistral_client else '❌ MISTRAL_KEY não encontrada'}")


In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  2. CONFIGURAÇÃO                                                 ║
# ╚══════════════════════════════════════════════════════════════════╝

# ── Vídeo/capítulo (mesmo padrão dos outros notebooks do projeto) ──────────
NOME_ORACAO = "40_Matt_02"
PASTA_DRIVE_RAIZ = "narrated_video"
IDIOMA_MESTRE = "en"
NOME_LEGENDA_MESTRE = "40_Matt_02_edge_en.srt"
CAPITULO = 2

# Cole aqui o texto do capítulo com os números de versículo isolados no
# meio do fluxo (mesmo formato usado no indicador "Matt 2:4"):
TEXTO_VERSICULOS = """"""

# ── Biblioteca a usar: "video" ou "imagem" ──────────────────────────────────
TIPO_FONTE = "video"

ID_PLANILHA_VIDEOS = "1bF7hnGSY7AALm4ZAS5owWNpiSTdgArW4ahAuVZaHPL0"
NOME_ABA_VIDEOS = "pixabay_stock"

ID_PLANILHA_IMAGENS = ""  # cole o ID da planilha de imagens aqui
NOME_ABA_IMAGENS = "image-stock"

# ── Modelos de IA (texto só — bem mais barato que a descrição de cena,     ──
# ── que usa imagem) ─────────────────────────────────────────────────────────
MODELO_GROQ = "qwen/qwen3.6-27b"
MODELO_MISTRAL = "mistral-small-latest"
DELAY_SEGUNDOS = 2
MAX_TOKENS_RESPOSTA = 300

# ── Anti-repetição ──────────────────────────────────────────────────────────
DIST_MIN_REPETICAO = 3    # nao repete o mesmo video/imagem em versiculos a menos de N de distancia...
MARGEM_PARA_REPETIR = 2   # ...a nao ser que a alternativa mais proxima perca por essa margem de score

print("=" * 60)
print("⚙️  CONFIGURAÇÃO")
print("=" * 60)
print(f"   Vídeo:        {NOME_ORACAO}")
print(f"   Capítulo:     {CAPITULO}")
print(f"   Fonte:        {TIPO_FONTE}")
print(f"   Versículos:   {'(vazio! preencha TEXTO_VERSICULOS)' if not TEXTO_VERSICULOS.strip() else 'preenchido'}")
print("=" * 60)


In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  3. INICIALIZAR — carrega a legenda mestre + abre a biblioteca   ║
# ╚══════════════════════════════════════════════════════════════════╝
from config import PipelineConfig
from srt_utils import ler_srt, texto_por_versiculo
from match_pipeline import carregar_biblioteca, gerar_sugestoes_match

config = PipelineConfig(
    NOME_ORACAO=NOME_ORACAO,
    PASTA_DRIVE_RAIZ=PASTA_DRIVE_RAIZ,
    IDIOMA_MESTRE=IDIOMA_MESTRE,
    NOME_LEGENDA_MESTRE=NOME_LEGENDA_MESTRE,
)

# Baixa a legenda mestre direto do Drive (mesma pasta dos outros notebooks)
import shutil as _shutil
from drive_utils import DriveClient
_drive = DriveClient.get()
_destino_mestre = Path(config.nome_legenda_mestre)
_drive.download(config.pasta_oracao, config.nome_legenda_mestre, _destino_mestre)
legendas_mestre = ler_srt(_destino_mestre)
print(f"✅ Legenda mestre carregada: {len(legendas_mestre)} blocos")

# Extrai o texto de cada versículo (sem timing — só o conteúdo)
if not TEXTO_VERSICULOS.strip():
    raise ValueError("TEXTO_VERSICULOS está vazio — cole o texto do capítulo na célula de Configuração.")
versiculos_texto = texto_por_versiculo(TEXTO_VERSICULOS)
print(f"✅ {len(versiculos_texto)} versículos extraídos do texto colado")

# Abre a planilha certa conforme TIPO_FONTE
if TIPO_FONTE == "video":
    id_planilha, nome_aba, coluna_url = ID_PLANILHA_VIDEOS, NOME_ABA_VIDEOS, "url"
elif TIPO_FONTE == "imagem":
    if not ID_PLANILHA_IMAGENS:
        raise ValueError("TIPO_FONTE='imagem' mas ID_PLANILHA_IMAGENS não foi preenchido.")
    id_planilha, nome_aba, coluna_url = ID_PLANILHA_IMAGENS, NOME_ABA_IMAGENS, "Imagem"
else:
    raise ValueError(f"TIPO_FONTE inválido: {TIPO_FONTE!r} (use 'video' ou 'imagem')")

sheet = gc.open_by_key(id_planilha).worksheet(nome_aba)
linhas_planilha = sheet.get_all_records()
biblioteca = carregar_biblioteca(linhas_planilha, coluna_url=coluna_url)
print(f"✅ Biblioteca ({TIPO_FONTE}): {len(linhas_planilha)} linhas na planilha, "
      f"{len(biblioteca)} já têm Tags_Biblia_PT preenchida (só essas entram no match)")


In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  4. LISTA FECHADA DE TAGS_BIBLIA (mesma do notebook de descrição) ║
# ╚══════════════════════════════════════════════════════════════════╝
LISTA_TAGS_BIBLIA = ("criação, jardim do éden, queda, dilúvio, arca de noé, torre de babel, "
"chamado de abraão, aliança, sacrifício de isaque, jacó e esaú, escada de jacó, josé e os "
"irmãos, sonhos proféticos, escravidão no egito, sarça ardente, pragas do egito, travessia do "
"mar vermelho, maná no deserto, dez mandamentos, monte sinai, bezerro de ouro, tabernáculo, "
"arca da aliança, peregrinação no deserto, serpente de bronze, terra prometida, queda de "
"jericó, juízes, sansão, gideão, débora, rute e noemi, davi e golias, davi e saul, reino de "
"davi, sabedoria de salomão, templo de salomão, reino dividido, elias no monte carmelo, carro "
"de fogo, eliseu, exílio babilônico, daniel na cova dos leões, fornalha ardente, jonas e o "
"grande peixe, ester, sofrimento de jó, salmos e louvor, provérbios e sabedoria, reconstrução "
"do templo, profecia messiânica, anunciação, natividade, magos do oriente, estrela de belém, "
"fuga para o egito, apresentação no templo, batismo de jesus, tentação no deserto, chamado dos "
"discípulos, sermão da montanha, bem-aventuranças, milagre de cura, multiplicação dos pães, "
"tempestade acalmada, jesus anda sobre as águas, parábola do semeador, parábola do filho "
"pródigo, parábola do bom samaritano, ovelha perdida, transfiguração, ressurreição de lázaro, "
"entrada triunfal em jerusalém, última ceia, getsêmani, prisão e julgamento, crucificação, "
"ressurreição de jesus, tumba vazia, estrada de emaús, ascensão, pentecostes, conversão de "
"paulo, viagens missionárias, igreja primitiva, perseguição dos cristãos, cartas apostólicas, "
"apocalipse, pastor e ovelhas, boas novas, anjo mensageiro, profeta, rei, sacerdote, juízo, "
"misericórdia divina, aliança renovada, êxodo espiritual, batalha espiritual, jornada de fé, "
"provação, milagre, cura, ressurreição, segunda vinda, reino de deus, cordeiro de deus, luz "
"do mundo")

print(f"{len(LISTA_TAGS_BIBLIA.split(','))} temas na lista fechada")


In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  5. RODAR O MATCH                                                ║
# ╚══════════════════════════════════════════════════════════════════╝
if not (groq_client or mistral_client):
    raise RuntimeError("Nenhuma API disponível — confira GROQ_KEY / MISTRAL_KEY nos Secrets do Colab")

resultados = gerar_sugestoes_match(
    versiculos_texto, biblioteca, LISTA_TAGS_BIBLIA,
    groq_client, mistral_client, MODELO_GROQ, MODELO_MISTRAL,
    dist_min_repeticao=DIST_MIN_REPETICAO, margem_para_repetir=MARGEM_PARA_REPETIR,
    delay_segundos=DELAY_SEGUNDOS, max_tokens=MAX_TOKENS_RESPOSTA,
)

com_match = sum(1 for r in resultados if not r["sem_opcao"])
print(f"\n{'='*60}")
print(f"✅ {len(resultados)} versículos processados")
print(f"   Com sugestão:  {com_match}")
print(f"   Sem opção:     {len(resultados) - com_match}")
print(f"{'='*60}")


In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  6. REVISAR OS RESULTADOS                                        ║
# ╚══════════════════════════════════════════════════════════════════╝
for r in resultados:
    cap_v = f"{CAPITULO}:{r['versiculo']}"
    if r["sem_opcao"]:
        print(f"❌ {cap_v:6s} SEM OPÇÃO — busque por: {', '.join(r['palavras_chave']) or '(nenhuma sugestão)'}")
    else:
        print(f"✅ {cap_v:6s} [{r['id']}] {r['titulo']}  (score={r['score']}, bateu em: {', '.join(r['tags_batidas'])})")

# Salva os resultados num JSON, pra usar depois (ex: alimentar o baixar_clipes())
nome_arquivo = f"match_{NOME_ORACAO}_cap{CAPITULO}.json"
with open(nome_arquivo, "w", encoding="utf-8") as f:
    json.dump(resultados, f, ensure_ascii=False, indent=2)
print(f"\n💾 Resultados salvos em {nome_arquivo}")

from google.colab import files
files.download(nome_arquivo)
